In [14]:
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from configuration.config import *

graph = Neo4jGraph(url=NEO4J_CONFIG['uri'], username=NEO4J_CONFIG['auth'][0], password=NEO4J_CONFIG['auth'][1])


In [7]:
print(graph.schema)

Node properties:
Category1 {id: INTEGER, name: STRING}
Category2 {id: INTEGER, name: STRING}
Category3 {id: INTEGER, name: STRING}
BaseAttrValue {id: INTEGER, name: STRING}
SPU {id: INTEGER, name: STRING}
SKU {id: INTEGER, name: STRING}
SaleAttrValue {id: INTEGER, name: STRING}
BaseAttr {id: INTEGER, name: STRING}
SaleAttr {id: INTEGER, name: STRING}
BaseTrademark {id: INTEGER, name: STRING}
Relationship properties:

The relationships:
(:Category1)-[:Have]->(:BaseAttr)
(:Category2)-[:Belong]->(:Category1)
(:Category2)-[:Have]->(:BaseAttr)
(:Category3)-[:Belong]->(:Category2)
(:Category3)-[:Have]->(:BaseAttr)
(:SPU)-[:Belong]->(:BaseTrademark)
(:SPU)-[:Belong]->(:Category3)
(:SPU)-[:Have]->(:SaleAttr)
(:SKU)-[:Have]->(:SaleAttrValue)
(:SKU)-[:Have]->(:BaseAttrValue)
(:SKU)-[:Belong]->(:SPU)
(:BaseAttr)-[:Have]->(:BaseAttrValue)
(:SaleAttr)-[:Have]->(:SaleAttrValue)


In [11]:
import os
from langchain_deepseek import ChatDeepSeek

# llm = ChatDeepSeek(
#     model="deepseek-v4-pro",  # 或 "deepseek-reasoner"
#     api_key=os.getenv("DEEPSEEK_API_KEY"),
#     temperature=0.7,
#     max_tokens=1024,
#     timeout=30,
#     max_retries=2
# )
# resp = llm.invoke("什么是大语言模型？")
# print(resp.content)

我们用一个生活中的比喻来理解大语言模型。

想象一下，有一个人，他一生唯一做的事情就是**读书**。他读遍了人类历史上所有的书籍、网页、代码、文章、对话记录……数量大到几辈子都读不完。

他读的时候，不一定是去“理解”书里的情感或深刻哲理，而是专注于做一件事：**学习并记忆人类语言的万千模式**。

-   他知道“因为”后面通常会跟着“所以”。
-   他知道“苹果”不仅是一种水果，还经常和“手机”这个词一起出现。
-   他知道“今天天气真不错”这句话，后面可以接“我们去公园吧”，也可以接“适合晒被子”。

这个人，就是一个**大语言模型**的精髓。

---

### 到底什么是大语言模型？

从技术上说，大语言模型，简称LLM，是一种**基于海量文本数据训练出来的、具有巨大参数量的深度学习人工智能模型**。

别被术语吓到，我们拆开来看看这三个关键词：

-   **语言模型**：它的核心任务就是理解和生成人类的自然语言。它的工作原理听起来简单，就是**预测下一个词**——“今天天气真___”后面，它通过计算给出“好”、“不错”、“热”等词的可能性高低。

-   **大**：这个词有两层含义。
    1.  **训练数据巨大**：它“读”过的文字，是整个互联网级别的，可能达到了TB甚至PB级别。这就像它见过无数的世面。
    2.  **模型参数巨大**：参数可以看作模型大脑中神经元的连接点，数量越多，理论上能学习并存储的模式就越复杂、越精细。现在的LLM，参数动辄上百亿、千亿级别。

-   **模型**：你可以理解为一个由无数个数学方程组成的、极其复杂的函数。它并不是真正地“思考”，而是通过输入一段文字，经过这些方程的层层计算，最终输出一个它认为最可能的结果。

---

### 它为什么如此强大？—— 一个比喻：书本空间的拼图

可以把大语言模型想象成一个高维的“文字地图”或“词语的星空”。

1.  **词语向量化**：训练过程中，它把每个词都转换成了空间中的一个个点（数学上叫“向量”）。

    -   一个非常神奇的现象是，这个空间里，**语义相近的词，在空间中的位置也近**。
    -   “国王”和“王后”的距离，与“男人”和“女人”的距离是相似的。
    -   “北京 - 中国 ≈ 巴黎 - 法国”。

2.  **涌现的能力

In [23]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI

load_dotenv()
# LLM
llm = ChatOpenAI(
    model="deepseek-v4-pro",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1",  # 关键
    temperature=0.7,
    # max_tokens=1024
)
chain = GraphCypherQAChain.from_llm(llm=llm, graph=graph, allow_dangerous_requests=True,  verbose=True)

result = chain.invoke({"query": "Apple有哪些产品？"})
print(result)




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (:BaseTrademark {name: 'Apple'})<-[:Belong]-(spu:SPU)
RETURN spu.name AS productName

Full Context:
[]

> Finished chain.
{'query': 'Apple有哪些产品？', 'result': '对不起，我不知道这个问题的答案。'}
